# 04 — SHAP Explainability

This notebook explains the final models from notebook 03 with SHAP (SHapley Additive exPlanations).

- **RQ1:** Which genetic features drive the predictions, and are they the established resistance
  mechanisms (gyrA S83/D87, parC S80/E84, qnr genes)?
- **RQ2:** Do the models also rely on substitutions that are not resistance mechanisms but mark
  particular lineages?

SHAP values are computed on the held-out test isolates. A SHAP value shows how much a feature pushed
one isolate's prediction towards resistant (positive) or susceptible (negative). SHAP describes the
model's behaviour; it is not evidence of biological causation.

## 1. Settings and data

In [ ]:
from pathlib import Path

RESULTS_DIR = Path("results")
MODELS_DIR = Path("models")
TOP_N = 20

In [ ]:
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

from src.amr_utils import CATEGORY_LABELS, feature_category

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
sns.set_theme(style="whitegrid")
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

datasets = joblib.load(MODELS_DIR / "datasets.joblib")
X_dev, X_test, y_test = datasets["X_dev"], datasets["X_test"], datasets["y_test"]
clades = datasets["clades"]

MODEL_FILES = {
    "Logistic Regression": "logistic_regression.joblib",
    "Random Forest": "random_forest.joblib",
    "XGBoost": "xgboost.joblib",
}
models = {name: joblib.load(MODELS_DIR / f) for name, f in MODEL_FILES.items()}
print(f"Test isolates: {len(X_test)} ({y_test.mean():.1%} resistant)")
print("shap", shap.__version__)

## 2. Compute SHAP values
Each model is a pipeline (rare-feature filter → model). SHAP is applied to the model step using the
features that passed the filter. Tree models use the exact TreeExplainer; Logistic Regression uses the
LinearExplainer with the development set as background. All values are on the log-odds scale.

In [ ]:
explanations = {}
for name, pipeline in models.items():
    X_dev_t = pipeline[:-1].transform(X_dev)
    X_test_t = pipeline[:-1].transform(X_test)
    estimator = pipeline[-1]
    if name == "Logistic Regression":
        explainer = shap.LinearExplainer(estimator, X_dev_t)
    else:
        explainer = shap.TreeExplainer(estimator)
    explanation = explainer(X_test_t)
    if explanation.values.ndim == 3:  # Random Forest returns one set per class
        explanation = explanation[:, :, 1]
    explanations[name] = explanation
    print(f"{name}: {X_test_t.shape[1]} features after filtering")

## 3. Global feature importance
Mean |SHAP| is the average size of a feature's contribution across the test isolates. Bars are
coloured by feature category.

In [ ]:
CATEGORY_COLOURS = {"target_qrdr": "#C44E52", "acquired": "#8172B3",
                    "efflux_truncation": "#DD8452", "other": "#8C8C8C"}

importance_tables = {}
for name, explanation in explanations.items():
    table = pd.DataFrame({
        "feature": explanation.feature_names,
        "mean_abs_shap": np.abs(explanation.values).mean(axis=0),
        "mean_shap_when_present": [
            explanation.values[explanation.data[:, i] == 1, i].mean() if (explanation.data[:, i] == 1).any() else np.nan
            for i in range(len(explanation.feature_names))],
    })
    table["category"] = table["feature"].map(feature_category)
    table = table.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    table["rank"] = table.index + 1
    importance_tables[name] = table
    table.to_csv(RESULTS_DIR / f"shap_importance_{name.lower().replace(' ', '_')}.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (name, table) in zip(axes, importance_tables.items()):
    top = table.head(TOP_N).iloc[::-1]
    ax.barh(top["feature"], top["mean_abs_shap"], color=top["category"].map(CATEGORY_COLOURS))
    ax.set_title(name)
    ax.set_xlabel("Mean |SHAP value|")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CATEGORY_COLOURS.values()]
fig.legend(handles, [CATEGORY_LABELS[k] for k in CATEGORY_COLOURS], loc="lower center", ncol=4)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(FIG_DIR / "shap_importance_all_models.png", dpi=200)
plt.show()

In [ ]:
for name, explanation in explanations.items():
    plt.figure()
    shap.plots.beeswarm(explanation, max_display=15, show=False)
    plt.title(f"{name}: SHAP values on the held-out test set")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"shap_beeswarm_{name.lower().replace(' ', '_')}.png", dpi=200, bbox_inches="tight")
    plt.show()

## 4. Do the important features match known resistance mechanisms? (RQ1)

In [ ]:
KEY_MUTATIONS = ["gyrA_S83L", "gyrA_D87N", "gyrA_D87Y", "parC_S80I", "parC_E84V", "parC_E84G",
                 "parE_S458A", "parE_L416F", "acq_qnrS", "acq_qnrB", "acq_aac(6')-Ib-cr"]

ranks = pd.DataFrame({
    name: table.set_index("feature")["rank"].reindex(KEY_MUTATIONS)
    for name, table in importance_tables.items()
})
ranks["present in test isolates"] = [int(X_test[f].sum()) if f in X_test else 0 for f in KEY_MUTATIONS]
print("Rank of established resistance mechanisms among all features (1 = most important; "
      "blank = removed by the rare-feature filter or not observed)")
display(ranks)

top_categories = pd.DataFrame({
    name: table.head(10)["category"].map(CATEGORY_LABELS).value_counts()
    for name, table in importance_tables.items()
}).fillna(0).astype(int)
print("\nCategories of the 10 most important features per model")
display(top_categories)

## 5. How much do the models rely on lineage markers? (RQ2)
The share of total |SHAP| attributed to each category shows how much of each model's reasoning
comes from known mechanisms vs other substitutions. For the most important 'other' features, lineage
concentration is the share of isolates carrying the feature that belong to a single lineage cluster:
values near 100% indicate a lineage marker.

In [ ]:
shares = pd.DataFrame({
    name: table.groupby("category")["mean_abs_shap"].sum() / table["mean_abs_shap"].sum()
    for name, table in importance_tables.items()
}).reindex(list(CATEGORY_COLOURS)).fillna(0).rename(index=CATEGORY_LABELS)
shares.to_csv(RESULTS_DIR / "shap_share_by_category.csv")

ax = (100 * shares.T).plot(kind="barh", stacked=True, figsize=(10, 3.5),
                           color=list(CATEGORY_COLOURS.values()))
ax.set_xlabel("% of total mean |SHAP|")
ax.set_title("Share of each model's attributions by feature category")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_share_by_category.png", dpi=200, bbox_inches="tight")
plt.show()
display((100 * shares).round(1))

In [ ]:
all_isolates = pd.concat([X_dev, X_test])
isolate_clades = clades.loc[all_isolates.index]

rows = []
for name, table in importance_tables.items():
    for _, row in table[table.category == "other"].head(10).iterrows():
        carriers = isolate_clades[all_isolates[row.feature] == 1]
        rows.append({
            "model": name, "feature": row.feature, "rank": row["rank"],
            "mean_abs_shap": row.mean_abs_shap, "isolates_with_feature": len(carriers),
            "lineages_with_feature": carriers.nunique(),
            "lineage_concentration_%": 100 * carriers.value_counts(normalize=True).iloc[0] if len(carriers) else np.nan,
        })
lineage_markers = pd.DataFrame(rows)
lineage_markers.to_csv(RESULTS_DIR / "shap_lineage_markers.csv", index=False)
display(lineage_markers.round(2))

## 6. Agreement between models
Spearman correlation of feature importance between models (features kept by both models).

In [ ]:
names = list(importance_tables)
agreement = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        merged = importance_tables[a].merge(importance_tables[b], on="feature")
        agreement.loc[a, b] = merged["mean_abs_shap_x"].corr(merged["mean_abs_shap_y"], method="spearman")
display(agreement.round(2))

## 7. Interaction between gyrA and parC mutations
High-level ciprofloxacin resistance usually needs a gyrA mutation plus a parC mutation. The dependence
plot shows whether the model's contribution for gyrA S83L changes with parC S80I.

In [ ]:
explanation = explanations["XGBoost"]
if {"gyrA_S83L", "parC_S80I"} <= set(explanation.feature_names):
    shap.plots.scatter(explanation[:, "gyrA_S83L"], color=explanation[:, "parC_S80I"], show=False)
    plt.title("XGBoost: SHAP value of gyrA S83L, coloured by parC S80I")
    plt.savefig(FIG_DIR / "shap_dependence_gyrA_parC.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("gyrA_S83L or parC_S80I was not kept by the XGBoost feature filter.")

## 8. Explaining individual predictions
Waterfall plots for one isolate of each outcome (XGBoost). They show how each feature moved the
prediction from the average (base value) to the final score for that isolate.

In [ ]:
explanation = explanations["XGBoost"]
prob = models["XGBoost"].predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)
outcomes = {
    "True positive (resistant, predicted resistant)": (y_test.to_numpy() == 1) & (pred == 1),
    "True negative (susceptible, predicted susceptible)": (y_test.to_numpy() == 0) & (pred == 0),
    "False negative (resistant, predicted susceptible)": (y_test.to_numpy() == 1) & (pred == 0),
    "False positive (susceptible, predicted resistant)": (y_test.to_numpy() == 0) & (pred == 1),
}
mic = datasets["metadata"]["measurement"]
for title, mask in outcomes.items():
    if not mask.any():
        print(f"{title}: none in the test set\n")
        continue
    i = int(np.flatnonzero(mask)[0])
    isolate = X_test.index[i]
    print(f"{title}\n  isolate {isolate}, MIC {mic[isolate]} mg/L, predicted probability {prob[i]:.2f}")
    shap.plots.waterfall(explanation[i], max_display=10, show=False)
    plt.title(title)
    plt.savefig(FIG_DIR / f"shap_waterfall_{title.split(' (')[0].lower().replace(' ', '_')}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

## 9. Summary for the dissertation

In [ ]:
for name, table in importance_tables.items():
    top5 = ", ".join(table.head(5)["feature"])
    known_share = 100 * (1 - shares.loc[CATEGORY_LABELS["other"], name])
    print(f"{name}\n  top 5 features: {top5}\n  share of attribution from known mechanisms: {known_share:.1f}%\n")